In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torch.nn as nn

In [3]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=0)

In [4]:
x = np.array([2.0, 1.0, 0.1])

In [5]:
output = softmax(x)
print("Softmax numpy output: ",output)

Softmax numpy output:  [0.65900114 0.24243297 0.09856589]


In [6]:
x = torch.tensor([2.0, 1.0, 0.1])

In [7]:
output = torch.softmax(x, dim=0)
print("Softmax tensor output: ", output)

Softmax tensor output:  tensor([0.6590, 0.2424, 0.0986])


In [8]:
def cross_entropy(actual, predicted):
    loss = -np.sum( actual * np.log(predicted) )
    return loss

In [9]:
# Y must be one hot encoded
# if class 0: [1,0,0]
# if class 1: [0,1,0]
# if class 2: [0,0,1]

In [10]:
Y = np.array([1,0,0])

In [11]:
y_pred_good = np.array([0.7, 0.2, 0.1])
y_pred_bad = np.array([0.1, 0.3, 0.6])

In [12]:
l1 = cross_entropy(Y, y_pred_good)
l2 = cross_entropy(Y, y_pred_bad)

In [13]:
print(f"Loss1 cross entropy (numpy): {l1}")
print(f"Loss2 cross entropy (numpy): {l2}")

Loss1 cross entropy (numpy): 0.35667494393873245
Loss2 cross entropy (numpy): 2.3025850929940455


# cross entropy in pytorch

In [14]:
loss = nn.CrossEntropyLoss()

Dikkatli olunması gerekenler
1. nn.CrossEntropyLoss fonksiyonu nn.LogSoftmax + nn.NLLLoss uyguluyor. yani çıkışta tekrar softmax yapmamıza gerek yok.
2. Y (sonuçlar) sınıf etiketi halinde olucak. One Hot Encoding yapılMAYACAK
3. Y_tahmin linear katmanın sonuçlarını yani ham değerleri (logits) içerecek. Softmax uygulanMAYACAK

In [15]:
Y = torch.tensor([0])

In [16]:
# size = nsample x nclass = 1 x 3
y_pred_good = torch.tensor([[2.0, 1.0, 0.1]])
y_pred_bad = torch.tensor([[0.5, 2, 0.3]])

In [17]:
l1 = loss(y_pred_good, Y)
l2 = loss(y_pred_bad, Y)

In [18]:
print(f"Loss1 cross entropy (tensor): {l1.item()}")
print(f"Loss2 cross entropy (tensor): {l2.item()}")

Loss1 cross entropy (tensor): 0.4170299470424652
Loss2 cross entropy (tensor): 1.840616226196289


In [19]:
_, prediction1 = torch.max(y_pred_good, 1)
_, prediction2 = torch.max(y_pred_bad, 1)

print(prediction1)
print(prediction2)

tensor([0])
tensor([1])


**2. örnek**

In [20]:
loss = nn.CrossEntropyLoss()

In [21]:
# 3 samples
Y = torch.tensor([2,0,1])
# ilk örneğin 2. sınıfı
# ikinci örneğin 0. sınıfı
# üçüncü örneğin 1. sınıfı imiş elimizdeki sonuçlar

In [22]:
#nsamples x nclasses = 3x3
y_pred_good = torch.tensor([ [0.1, 1.0, 5], [2.0, 0.7, 0.3], [0.5, 3, 0.3] ])
y_pred_bad = torch.tensor([ [3, 0.1, 0.3], [0.2, 2, 0.1], [0.5, 0.3, 4] ])

In [23]:
l1 = loss(y_pred_good, Y)
l2 = loss(y_pred_bad, Y)

In [24]:
print(f"loss1: {l1.item()}")
print(f"loss2: {l2.item()}")

loss1: 0.1799115389585495
loss2: 2.8808395862579346


In [25]:
_, prediction1 = torch.max(y_pred_good, 1)
_, prediction2 = torch.max(y_pred_bad, 1)

print(prediction1)
print(prediction2)

tensor([2, 0, 1])
tensor([0, 1, 2])


tensor([2, 0, 1])
tensor([0, 1, 2])
bu çıktıda iyi tahmin sonucunda zaten yüksek değerin olduğu yeri verdiğimiz için loss ve tahmin istediğimiz gibi
kötü de yanlış çıktı

In [26]:
# hata ne kadar düşük çıkarsa o kadar doğru tahmin etmişiz demektir mantıken
# 2. örnekte y_pred_good' da her örnek için doğru yerde ham değerin değerini yüksek yazdım y_pred_bad'de ise bilerek kötü yazdım. loss büyük çıktı.

# Multi Class Problem 
hangi hayvan diye sorarız model bize ihtimalleri sıralar
0.95 köpek
0.05 kedi

In [27]:
class NeuralNetMC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNetMC, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, num_classes) #num_classes = çıktı boyutu

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # sonda softmax yapmıyoruz
        return out

In [28]:
model = NeuralNetMC(input_size=28*28, hidden_size=5, num_classes=3)
criteiron = nn.CrossEntropyLoss() # softmax uygular en son bu fonksiyon

# Binary Classification Problem
bu bir köpek mi diye sorarız. model bize evet veya hayır der. 0.90 evet şeklinde tek çıkış vardır

In [29]:
class NeuralBC(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(NeuralBC, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, 1) # tek çıkış var bu yüzden 1

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)

        # sonda sigmoid uyguluyoruz
        y_pred = torch.sigmoid(out)

        return y_pred

In [30]:
model = NeuralBC(input_size=28*28, hidden_size=5)
criteiron = nn.BCELoss()